In [1]:
# ============================================================
# IMPORTS — RAG DOCUMENT INGESTION PIPELINE
# ============================================================

# --- Python standard library ---

from pathlib import Path
# Path = lucrează cu fișiere și directoare într-un mod mai sigur decât string-uri simple.

import hashlib
# hashlib = calculează hash-ul unui fișier; îl folosim pentru identificarea duplicatelor.

from time import perf_counter
# perf_counter = măsoară precis durata procesării unui document sau a unei execuții.

import os
# os = acces la variabilele de mediu, de exemplu user/parolă PostgreSQL.


# --- Environment variables ---

from dotenv import load_dotenv
# load_dotenv = citește variabilele din fișierul local .getenv și le pune în mediul procesului.


# --- PostgreSQL ---

import psycopg as pg
# psycopg = client Python pentru conectarea și executarea comenzilor SQL în PostgreSQL.

from psycopg.types.json import Jsonb
# Jsonb = permite trimiterea unui dicționar Python într-o coloană PostgreSQL de tip JSONB.


# --- Docling: document conversion ---

from docling.document_converter import DocumentConverter, PdfFormatOption
# DocumentConverter = obiectul principal care transformă PDF/DOCX/HTML etc. în DoclingDocument.
# PdfFormatOption = permite configurarea modului în care sunt procesate PDF-urile.

from docling.datamodel.base_models import InputFormat
# InputFormat = identifică tipul documentului procesat, de exemplu PDF.

from docling.datamodel.pipeline_options import PdfPipelineOptions
# PdfPipelineOptions = configurează OCR, tabele, imagini, modele locale etc.

from docling.datamodel.accelerator_options import AcceleratorOptions
# AcceleratorOptions = configurează CPU/GPU și numărul de thread-uri folosite de Docling.


# --- Docling document structure ---

from docling_core.types.doc import DoclingDocument
# DoclingDocument = reprezentarea structurată a documentului după conversie:
# text, tabele, imagini, structură, provenance.


# --- Chunking ---

from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
# HybridChunker = împarte DoclingDocument în bucăți potrivite pentru RAG,
# păstrând cât posibil structura semantică a documentului.

from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
# HuggingFaceTokenizer = adaptor între tokenizer-ul Hugging Face și chunker-ul Docling.


# --- Hugging Face tokenizer ---

from transformers import AutoTokenizer
# AutoTokenizer = încarcă tokenizer-ul modelului de embeddings, în cazul nostru BGE-M3.


# --- OpenAI-compatible API client ---

from openai import OpenAI
# OpenAI = client Python folosit pentru a comunica cu serverele vLLM
# prin API compatibil OpenAI: embeddings și ulterior LLM.

In [2]:
# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/home/bm/formation/RAG/cours-rag/"
    "drive-download-20261008T064617Z-1-001/rag/demo"
)
# PROJECT_ROOT = folderul principal al exercițiului RAG.

CORPUS = PROJECT_ROOT / "data" / "veltis"
# CORPUS = documentele sursă care vor intra în pipeline.

EXPORT_FOLDER = PROJECT_ROOT / "export_folder"
# EXPORT_FOLDER = locul unde vom salva rezultatele Docling:
# Markdown, JSON etc.

MODEL_PATH = Path("/home/bm/models/rag/embedder/bge-m3")
# MODEL_PATH = modelul local BGE-M3 folosit pentru tokenizer și embeddings.

SCHEMA_PATH = PROJECT_ROOT / "queries" / "schema.sql"
# SCHEMA_PATH = definiția SQL pentru tabelele run și document.

ENV_FILE = PROJECT_ROOT / ".getenv"
# ENV_FILE = credențialele PostgreSQL locale.
# Nu îl afișăm și nu îl punem în Git.


# ============================================================
# VERIFY PATHS
# ============================================================

print("PROJECT_ROOT :", PROJECT_ROOT.exists())
print("CORPUS       :", CORPUS.exists())
print("EXPORT_FOLDER:", EXPORT_FOLDER.exists())
print("MODEL_PATH   :", MODEL_PATH.exists())
print("SCHEMA_PATH  :", SCHEMA_PATH.exists())
print("ENV_FILE     :", ENV_FILE.exists())

PROJECT_ROOT : True
CORPUS       : True
EXPORT_FOLDER: True
MODEL_PATH   : True
SCHEMA_PATH  : True
ENV_FILE     : True


In [3]:
# ============================================================
# CELL 3 — INSPECT THE VELTIS CORPUS
# ============================================================

# Collect all files in the corpus directory.
files = sorted(p for p in CORPUS.rglob("*") if p.is_file())

# Display the number of files found.
print(f"Total files: {len(files)}")

# Display each file's name and extension.
for file in files:
    print(f"{file.name:55} | {file.suffix.lower()}")

Total files: 17
4.png                                                   | .png
8.png                                                   | .png
grille_salariale_2026.pdf                               | .pdf
guide_onboarding.pdf                                    | .pdf
4.png                                                   | .png
8.png                                                   | .png
intranet_actualites.html                                | .html
intranet_faq_rh.html                                    | .html
newsletter_le_fil_2026-02.pdf                           | .pdf
organigramme_et_processus_production.pdf                | .pdf
plan_strategique_2026-2028.pptx                         | .pptx
politique_teletravail_2023.docx                         | .docx
politique_teletravail_2025.docx                         | .docx
politique_teletravail_2025.pdf                          | .pdf
presentation_veltis.pdf                                 | .pdf
pv_conseil_administration_2026-03-

In [4]:
# ============================================================
# CHECKPOINT 3A — INVESTIGATE REPEATED PNG FILENAMES
# ============================================================

for file in files:
    if file.suffix.lower() == ".png":
        print("Path:", file.relative_to(CORPUS))
        print("Size:", file.stat().st_size, "bytes")
        print()
        

Path: 4.png
Size: 49518 bytes

Path: 8.png
Size: 25759 bytes

Path: images/4.png
Size: 49518 bytes

Path: images/8.png
Size: 25759 bytes



In [5]:
# ============================================================
# CELL 4 — CORPUS INVENTORY
# ============================================================

from collections import Counter

# Count files by extension
formats = Counter(file.suffix.lower() for file in files)

print("DOCUMENT FORMATS:")
for extension, count in sorted(formats.items()):
    print(f"{extension or '[no extension]':10} : {count}")

print("\nFILES AND SIZES:")
for file in files:
    size_kb = file.stat().st_size / 1024
    print(f"{file.name:55} | {size_kb:8.1f} KB")

DOCUMENT FORMATS:
.docx      : 2
.html      : 2
.pdf       : 8
.png       : 4
.pptx      : 1

FILES AND SIZES:
4.png                                                   |     48.4 KB
8.png                                                   |     25.2 KB
grille_salariale_2026.pdf                               |      4.5 KB
guide_onboarding.pdf                                    |      6.4 KB
4.png                                                   |     48.4 KB
8.png                                                   |     25.2 KB
intranet_actualites.html                                |      2.8 KB
intranet_faq_rh.html                                    |      3.3 KB
newsletter_le_fil_2026-02.pdf                           |    138.3 KB
organigramme_et_processus_production.pdf                |     92.7 KB
plan_strategique_2026-2028.pptx                         |     60.5 KB
politique_teletravail_2023.docx                         |     37.9 KB
politique_teletravail_2025.docx                  

In [6]:
# ============================================================
# CELL 4 — SELECT DOCUMENTS FOR INGESTION
# ============================================================

# Document formats supported in our exercise
SUPPORTED_FORMATS = {".pdf", ".docx", ".pptx", ".html"}

# Select documents from the existing inventory
documents = [
    file for file in files
    if file.suffix.lower() in SUPPORTED_FORMATS
]

print(f"Total files found : {len(files)}")
print(f"Documents selected: {len(documents)}")
print(f"Other files       : {len(files) - len(documents)}")

for document in documents:
    print(document.relative_to(CORPUS))

Total files found : 17
Documents selected: 13
Other files       : 4
grille_salariale_2026.pdf
guide_onboarding.pdf
intranet_actualites.html
intranet_faq_rh.html
newsletter_le_fil_2026-02.pdf
organigramme_et_processus_production.pdf
plan_strategique_2026-2028.pptx
politique_teletravail_2023.docx
politique_teletravail_2025.docx
politique_teletravail_2025.pdf
presentation_veltis.pdf
pv_conseil_administration_2026-03-12_scan.pdf
rapport_annuel_2025.pdf


In [7]:
# ============================================================
# CELL 5 — INSPECT POSTGRESQL SCHEMA
# ============================================================

# Read Romain's SQL schema without executing it.
schema_sql = SCHEMA_PATH.read_text(encoding="utf-8")

# Display the SQL instructions.
print(schema_sql)

CREATE TABLE IF NOT EXISTS run(
    run_id SERIAL CONSTRAINT PK_RUN_ID PRIMARY KEY,
    start_time TIMESTAMPTZ DEFAULT NOW(),
    end_time TIMESTAMPTZ,
    doc_processed INT,
    doc_ignored INT,
    doc_failed INT,
    parameters JSONB
);

CREATE TABLE IF NOT EXISTS document (
    document_id SERIAL CONSTRAINT PK_DOCUMENT_ID PRIMARY KEY,
    run_id INT CONSTRAINT FK_DOCUMENT_RUN REFERENCES run(run_id),
    file_path VARCHAR(255),
    export_folder VARCHAR(255),
    file_name VARCHAR(255),
    file_format_ VARCHAR(255),
    file_hash VARCHAR(255),
    status_ VARCHAR(255),
    page_count INT,
    text_count INT,
    table_count INT,
    image_count INT,
    process_time FLOAT,
    error VARCHAR(255)
)


In [8]:
# ============================================================
# CELL 6 — LOAD POSTGRESQL CONFIGURATION
# ============================================================

# Load configuration from the existing .getenv file.
load_dotenv(ENV_FILE)

# Check the configuration without displaying passwords.
print("Environment file loaded:", ENV_FILE.exists())

# Show variable names only, never their values.
env_keys = []

for line in ENV_FILE.read_text(encoding="utf-8").splitlines():
    line = line.strip()

    if line and not line.startswith("#") and "=" in line:
        key = line.split("=", 1)[0].strip()
        env_keys.append(key)

print("Configured variable names:")
for key in env_keys:
    print(" -", key)

Environment file loaded: True
Configured variable names:
 - POSTGRES_USER
 - POSTGRES_PASSWORD


In [12]:
# ============================================================
# CELL 7 — TEST POSTGRESQL CONNECTION
# ============================================================

# Read credentials from environment variables
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")

# Connection parameters
DB_HOST = "127.0.0.1"
DB_PORT = 5432
DB_NAME = "postgres"

# Test connection without modifying the database
try:
    with pg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=db_user,
        password=db_password,
        connect_timeout=5
    ) as conn:

        result = conn.execute(
            "SELECT current_database(), current_user, version()"
        ).fetchone()

        print("Connection: SUCCESS")
        print("Database:", result[0])
        print("User:", result[1])
        print("PostgreSQL:", result[2].split(",")[0])

except Exception as error:
    print("Connection: FAILED")
    print("Error type:", type(error).__name__)
    print("Details:", error)

Connection: SUCCESS
Database: postgres
User: bstromMB
PostgreSQL: PostgreSQL 17.11 (Debian 17.11-1.pgdg13+2) on x86_64-pc-linux-gnu


In [13]:
# ============================================================
# CELL 8 — CHECK POSTGRESQL PORTS
# ============================================================

import socket  # Permet de tester une connexion réseau TCP.

DB_HOST = "127.0.0.1"  # La machine Z4 elle-même.

PORTS = [5432, 5433]  # Deux ports PostgreSQL à vérifier.

for port in PORTS:  # Tester les ports l'un après l'autre.

    # Créer une connexion temporaire, avec un délai de 2 secondes.
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:

        sock.settimeout(2)  # Éviter d'attendre indéfiniment.

        result = sock.connect_ex((DB_HOST, port))

        # Le code 0 signifie qu'une connexion TCP est possible.
        if result == 0:
            print(f"Port {port}: OPEN")
        else:
            print(f"Port {port}: CLOSED")

Port 5432: OPEN
Port 5433: CLOSED


In [11]:
# ============================================================
# CELL 9 — CONNECT TO POSTGRESQL
# ============================================================

DB_PORT = 5432  # Use the open port identified in Cell 8.

try:
    # Open a connection using the existing configuration.
    with pg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=db_user,
        password=db_password,
        connect_timeout=5
    ) as conn:

        # Ask PostgreSQL to identify the current database and user.
        result = conn.execute(
            "SELECT current_database(), current_user, version()"
        ).fetchone()

        # Display connection information.
        print("Connection: SUCCESS")
        print("Database:", result[0])
        print("User:", result[1])
        print("PostgreSQL:", result[2].split(",")[0])

except Exception as error:
    # Display the error without exposing credentials.
    print("Connection: FAILED")
    print("Error type:", type(error).__name__)
    print("Details:", error)

Connection: SUCCESS
Database: postgres
User: bstromMB
PostgreSQL: PostgreSQL 17.11 (Debian 17.11-1.pgdg13+2) on x86_64-pc-linux-gnu


In [14]:
# ============================================================
# CELL 10 — INSPECT EXISTING POSTGRESQL TABLES
# ============================================================

# Connect using the parameters verified in Cell 9.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    # Retrieve tables from the public schema.
    query = """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_type = 'BASE TABLE'
        ORDER BY table_name;
    """

    # Execute the SQL query and retrieve all results.
    tables = conn.execute(query).fetchall()

    # Display the existing tables.
    print(f"Existing tables: {len(tables)}")

    for table in tables:
        print(" -", table[0])

Existing tables: 0


In [15]:
# ============================================================
# CELL 11 — CREATE POSTGRESQL TABLES
# ============================================================

# Connect to the PostgreSQL database.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    # Execute Romain's SQL schema loaded in Cell 5.
    conn.execute(schema_sql)

    # Save the changes permanently.
    conn.commit()

    # Verify that both tables were created.
    result = conn.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_name IN ('run', 'document')
        ORDER BY table_name;
    """).fetchall()

    # Display the result.
    print("Tables created or verified:", len(result))

    for row in result:
        print(" -", row[0])

Tables created or verified: 2
 - document
 - run


In [16]:
# ============================================================
# CELL 12 — REGISTER A NEW PIPELINE RUN
# ============================================================

# Configuration describing this execution.
run_parameters = {
    "project": "Veltis RAG",
    "tool": "Docling",
    "total_documents": len(documents)
}

# Connect to PostgreSQL.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    # Insert one execution record.
    result = conn.execute(
        """
        INSERT INTO run (parameters)
        VALUES (%s)
        RETURNING run_id;
        """,
        (Jsonb(run_parameters),)
    )

    # Retrieve the generated identifier.
    run_id = result.fetchone()[0]

    # Save the transaction.
    conn.commit()

print("Pipeline run created successfully")
print("Run ID:", run_id)
print("Documents planned:", len(documents))

Pipeline run created successfully
Run ID: 1
Documents planned: 13


In [17]:
# ============================================================
# CELL 13 — INSPECT THE EXISTING RUN
# ============================================================

# Open a connection to PostgreSQL.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    # Read the record created in Cell 12.
    result = conn.execute(
        """
        SELECT
            run_id,
            start_time,
            end_time,
            doc_processed,
            doc_ignored,
            doc_failed,
            parameters
        FROM run
        WHERE run_id = %s;
        """,
        (run_id,)
    ).fetchone()

    # Display the stored values.
    if result is not None:
        columns = [
            "run_id",
            "start_time",
            "end_time",
            "doc_processed",
            "doc_ignored",
            "doc_failed",
            "parameters"
        ]

        for column, value in zip(columns, result):
            print(f"{column}: {value}")

    else:
        print("Run not found.")

run_id: 1
start_time: 2026-10-12 16:40:27.797124+00:00
end_time: None
doc_processed: None
doc_ignored: None
doc_failed: None
parameters: {'tool': 'Docling', 'project': 'Veltis RAG', 'total_documents': 13}


In [18]:
# ============================================================
# CELL 14 — REUSE AN UNFINISHED RUN
# ============================================================

# Identify the project whose unfinished run we want to reuse.
PROJECT_NAME = "Veltis RAG"

# Connect to PostgreSQL using our verified settings.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    # Look for unfinished runs belonging to this project.
    existing_runs = conn.execute(
        """
        SELECT run_id
        FROM public.run
        WHERE end_time IS NULL
          AND parameters->>'project' = %s
        ORDER BY run_id DESC;
        """,
        (PROJECT_NAME,)
    ).fetchall()

    # Reuse the run if exactly one unfinished execution exists.
    if len(existing_runs) == 1:
        run_id = existing_runs[0][0]
        print(f"Reusing existing Run ID: {run_id}")

    # Do not create another run if several are unfinished.
    elif len(existing_runs) > 1:
        raise RuntimeError(
            "Multiple unfinished runs found. "
            "Inspect PostgreSQL before continuing."
        )

    # No unfinished run: stop rather than create one automatically.
    else:
        raise RuntimeError(
            "No unfinished run found. "
            "A new run must be created explicitly."
        )

Reusing existing Run ID: 1


In [19]:
# ============================================================
# CELL 15 — CHECK DOCUMENT PROCESSING STATUS
# ============================================================

# Connect to PostgreSQL.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    # Retrieve documents already registered for this run.
    rows = conn.execute(
        """
        SELECT file_name, file_hash, status_
        FROM public.document
        WHERE run_id = %s;
        """,
        (run_id,)
    ).fetchall()

# Display the current state.
print(f"Run ID: {run_id}")
print(f"Documents expected: {len(documents)}")
print(f"Document records found: {len(rows)}")

# Display existing records, if any.
for file_name, file_hash, status in rows:
    print(f"{file_name} | {status} | {file_hash}")

# Check whether processing records already exist.
if not rows:
    print("\nAll 13 documents are awaiting processing.")

Run ID: 1
Documents expected: 13
Document records found: 0

All 13 documents are awaiting processing.


In [20]:
# ============================================================
# CELL 16 — CALCULATE SHA-256 HASHES
# ============================================================

# Dictionary to store each document's hash.
document_hashes = {}

# Process each document selected in Cell 4.
for document in documents:

    # Create a SHA-256 hash calculator.
    sha256 = hashlib.sha256()

    # Open the document in binary reading mode.
    with document.open("rb") as file:

        # Read the file in blocks of 8192 bytes.
        for block in iter(lambda: file.read(8192), b""):

            # Add each block to the hash calculation.
            sha256.update(block)

    # Convert the result to a hexadecimal string.
    file_hash = sha256.hexdigest()

    # Associate the document path with its hash.
    document_hashes[document] = file_hash

    # Display the document name and the first 12 hash characters.
    print(f"{document.name:55} | {file_hash[:12]}")

# Verify the total number of hashed documents.
print(f"\nDocuments hashed: {len(document_hashes)}")

grille_salariale_2026.pdf                               | 356850c4f1c2
guide_onboarding.pdf                                    | cf22ac557686
intranet_actualites.html                                | aa20469763c8
intranet_faq_rh.html                                    | 965f9e45050b
newsletter_le_fil_2026-02.pdf                           | e78d0f7ce71e
organigramme_et_processus_production.pdf                | 886bffe323aa
plan_strategique_2026-2028.pptx                         | 748a2eeb47e0
politique_teletravail_2023.docx                         | a2827f55b6ae
politique_teletravail_2025.docx                         | f9295e579ccd
politique_teletravail_2025.pdf                          | 700b6eac4cc1
presentation_veltis.pdf                                 | ad2de90af0f5
pv_conseil_administration_2026-03-12_scan.pdf           | 2f7cfe3bbd8e
rapport_annuel_2025.pdf                                 | 22968782738c

Documents hashed: 13


In [21]:
# ============================================================
# CELL 17 — CHECK HASHES AND PROCESSING STATUS
# ============================================================

# Connect to PostgreSQL and retrieve successful documents.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    rows = conn.execute(
        """
        SELECT file_hash
        FROM public.document
        WHERE run_id = %s
          AND status_ = 'success';
        """,
        (run_id,)
    ).fetchall()

# Convert the query results into a Python set.
successful_hashes = {row[0] for row in rows}

# Prepare our two document lists.
to_process = []
to_skip = []

# Compare each document's hash with the database records.
for document in documents:

    file_hash = document_hashes[document]

    if file_hash in successful_hashes:
        to_skip.append(document)
        print(f"SKIP    | {document.name}")

    else:
        to_process.append(document)
        print(f"PROCESS | {document.name}")

# Display the processing plan.
print("\nPROCESSING PLAN")
print("Run ID:", run_id)
print("Documents to process:", len(to_process))
print("Documents to skip:", len(to_skip))

PROCESS | grille_salariale_2026.pdf
PROCESS | guide_onboarding.pdf
PROCESS | intranet_actualites.html
PROCESS | intranet_faq_rh.html
PROCESS | newsletter_le_fil_2026-02.pdf
PROCESS | organigramme_et_processus_production.pdf
PROCESS | plan_strategique_2026-2028.pptx
PROCESS | politique_teletravail_2023.docx
PROCESS | politique_teletravail_2025.docx
PROCESS | politique_teletravail_2025.pdf
PROCESS | presentation_veltis.pdf
PROCESS | pv_conseil_administration_2026-03-12_scan.pdf
PROCESS | rapport_annuel_2025.pdf

PROCESSING PLAN
Run ID: 1
Documents to process: 13
Documents to skip: 0


In [22]:
# ============================================================
# CELL 18 — CONFIGURE DOCLING
# ============================================================

# Select our first test document.
test_document = CORPUS / "guide_onboarding.pdf"

# Configure the PDF processing pipeline.
pipeline_options = PdfPipelineOptions()

# Enable OCR to support scanned documents.
pipeline_options.do_ocr = True

# Enable table structure recognition.
pipeline_options.do_table_structure = True

# Create the document converter with our PDF settings.
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

# Verify that the source file exists.
print("Document:", test_document.name)
print("File exists:", test_document.is_file())
print("OCR enabled:", pipeline_options.do_ocr)
print("Table recognition:", pipeline_options.do_table_structure)
print("Converter ready:", converter is not None)

Document: guide_onboarding.pdf
File exists: True
OCR enabled: True
Table recognition: True
Converter ready: True


In [23]:
# ============================================================
# CELL 19 — CONVERT FIRST DOCUMENT WITH DOCLING
# ============================================================

# Start measuring processing time.
start_time = perf_counter()

# Convert the PDF using the converter configured in Cell 18.
conversion_result = converter.convert(test_document)

# Calculate processing duration in seconds.
processing_time = perf_counter() - start_time

# Retrieve the structured document produced by Docling.
docling_doc = conversion_result.document

# Export the extracted content to Markdown in memory.
markdown_text = docling_doc.export_to_markdown()

# Display conversion results.
print("Document:", test_document.name)
print("Conversion status:", conversion_result.status)
print(f"Processing time: {processing_time:.2f} seconds")
print("Extracted characters:", len(markdown_text))

# Display the first 1,000 characters for inspection.
print("\n--- EXTRACTED TEXT PREVIEW ---")
print(markdown_text[:1000])

[INFO] 2026-10-12 20:03:59,384 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-10-12 20:03:59,388 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-10-12 20:03:59,410 [RapidOCR] download_file.py:60: File exists and is valid: /home/bm/formation/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-10-12 20:03:59,411 [RapidOCR] main.py:50: Using /home/bm/formation/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-10-12 20:03:59,632 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-10-12 20:03:59,633 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-10-12 20:03:59,636 [RapidOCR] download_file.py:60: File exists and is valid: /home/bm/formation/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-10-12 20:03:59,636 [RapidOCR] main.py:50: Using /home/bm/formation/.venv/lib/python3.12/site-packages/rapidocr/models/

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/home/bm/formation/.venv/lib/python3.12/site-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1104.)
  return F.conv2d(


Document: guide_onboarding.pdf
Conversion status: ConversionStatus.SUCCESS
Processing time: 4.37 seconds
Extracted characters: 3781

--- EXTRACTED TEXT PREVIEW ---
## Guide d'accueil des nouveaux collaborateurs

Ressources humaines - version de janvier 2026

## 1. Bienvenue chez Veltis

Ce guide rassemble tout ce qu'il faut savoir pour vos premières semaines. Il ne remplace pas les échanges avec votre manager et votre parrain ou marraine : n'hésitez jamais à poser des questions. - Sophie Marchal, directrice des ressources humaines

## 2. Avant votre arrivée

## 2.1 Documents à fournir

- une copie de votre pièce d'identité ;
- votre numéro de compte bancaire pour le versement du salaire ;
- le formulaire de choix de l'assurance hospitalisation (famille ou individuelle) ;
- votre attestation de domicile si vous demandez l'indemnité vélo.

## 2.2 Votre parrain ou votre marraine

Chaque nouvelle personne est accompagnée par un parrain ou une marraine issu(e) d'un autre service, pendant se

In [ ]:
# ============================================================
# CELL 20A — CLEAN DOCLING EXPORT FOLDER
# ============================================================

# Verify the target directory.
print("Export folder:", EXPORT_FOLDER.resolve())

# Identify existing files.
existing_files = sorted(
    p for p in EXPORT_FOLDER.rglob("*") if p.is_file()
)

print(f"\nFiles found: {len(existing_files)}")

for file in existing_files:
    print(file.relative_to(EXPORT_FOLDER))

# Remove generated JSON and Markdown files only.
generated_files = [
    p for p in existing_files
    if p.suffix.lower() in {".json", ".md"}
]

print(f"\nGenerated files selected for deletion: {len(generated_files)}")

# Explicit safety switch: change to True after reviewing the list.
CONFIRM_DELETE = False

if CONFIRM_DELETE:
    for file in generated_files:
        file.unlink()
        print("DELETED:", file.relative_to(EXPORT_FOLDER))
else:
    print("\nDRY RUN — Nothing deleted.")
    print("Set CONFIRM_DELETE = True to perform the cleanup.")

Export folder: /home/bm/formation/RAG/cours-rag/drive-download-20261008T064617Z-1-001/rag/demo/export_folder

Files found: 34
4.png.json
4.png.md
8.png.json
8.png.md
grille_salariale_2026.pdf.json
grille_salariale_2026.pdf.md
guide_onboarding.pdf.json
guide_onboarding.pdf.md
images__4.png.json
images__4.png.md
images__8.png.json
images__8.png.md
intranet_actualites.html.json
intranet_actualites.html.md
intranet_faq_rh.html.json
intranet_faq_rh.html.md
newsletter_le_fil_2026-02.pdf.json
newsletter_le_fil_2026-02.pdf.md
organigramme_et_processus_production.pdf.json
organigramme_et_processus_production.pdf.md
plan_strategique_2026-2028.pptx.json
plan_strategique_2026-2028.pptx.md
politique_teletravail_2023.docx.json
politique_teletravail_2023.docx.md
politique_teletravail_2025.docx.json
politique_teletravail_2025.docx.md
politique_teletravail_2025.pdf.json
politique_teletravail_2025.pdf.md
presentation_veltis.pdf.json
presentation_veltis.pdf.md
pv_conseil_administration_2026-03-12_scan.pd

In [30]:
# ============================================================
# CELL 20 — EXPORT DOCLING DOCUMENT (CORRECTED)
# ============================================================

# Create the export directory if necessary.
EXPORT_FOLDER.mkdir(parents=True, exist_ok=True)

# Keep the complete original filename, including its extension.
source_name = test_document.name

# Build export paths.
json_path = EXPORT_FOLDER / f"{source_name}.json"
md_path = EXPORT_FOLDER / f"{source_name}.md"

# Prevent accidental overwriting.
if json_path.exists() or md_path.exists():
    raise FileExistsError(
        f"Export already exists for {source_name}"
    )

# Save the structured document.
docling_doc.save_as_json(json_path)

# Save readable Markdown.
md_path.write_text(markdown_text, encoding="utf-8")

# Verify both exports.
print("Source:", source_name)
print("JSON:", json_path.name, "| Exists:", json_path.is_file())
print("Markdown:", md_path.name, "| Exists:", md_path.is_file())

Source: guide_onboarding.pdf
JSON: guide_onboarding.pdf.json | Exists: True
Markdown: guide_onboarding.pdf.md | Exists: True


In [31]:
# ============================================================
# CELL 21 — VERIFY THE EXPORTED JSON
# ============================================================

# Reload the document from the saved JSON file.
reloaded_doc = DoclingDocument.load_from_json(json_path)

# Convert the reloaded document to Markdown for comparison.
reloaded_text = reloaded_doc.export_to_markdown()

# Compare the original and reloaded text.
text_matches = reloaded_text == markdown_text

# Display the verification results.
print("JSON file:", json_path.name)
print("JSON loaded successfully:", reloaded_doc is not None)
print("Original characters:", len(markdown_text))
print("Reloaded characters:", len(reloaded_text))
print("Text matches:", text_matches)

# Stop if the content differs.
if not text_matches:
    raise ValueError("Reloaded document differs from the original.")

JSON file: guide_onboarding.pdf.json
JSON loaded successfully: True
Original characters: 3781
Reloaded characters: 3781
Text matches: True


In [32]:
# ============================================================
# CELL 22 — REGISTER VERIFIED DOCUMENT IN POSTGRESQL
# ============================================================

# 1. Verify the export before registering it.
if not text_matches:
    raise ValueError("Document verification failed.")

if not json_path.is_file() or not md_path.is_file():
    raise FileNotFoundError("One or both exports are missing.")

# 2. Retrieve the SHA-256 calculated in Cell 16.
file_hash = document_hashes[test_document]

# 3. Open the PostgreSQL connection.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    # 4. Check whether this document was already registered successfully.
    existing = conn.execute(
        """
        SELECT document_id
        FROM public.document
        WHERE run_id = %s
          AND file_hash = %s
          AND status_ = 'success';
        """,
        (run_id, file_hash)
    ).fetchone()

    # 5. Insert only if no successful record exists.
    if existing:
        document_id = existing[0]
        print(f"SKIPPED: Document already registered (ID {document_id})")

    else:
        result = conn.execute(
            """
            INSERT INTO public.document (
                run_id,
                file_path,
                export_folder,
                file_name,
                file_format_,
                file_hash,
                status_,
                process_time
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            RETURNING document_id;
            """,
            (
                run_id,
                str(test_document),
                str(EXPORT_FOLDER),
                test_document.name,
                test_document.suffix.lower(),
                file_hash,
                "success",
                processing_time
            )
        )

        # 6. Retrieve the generated document identifier.
        document_id = result.fetchone()[0]

        print(f"REGISTERED: Document ID {document_id}")

    # 7. Commit the transaction.
    conn.commit()

print("Run ID:", run_id)
print("Document:", test_document.name)
print("SHA-256:", file_hash[:12])

REGISTERED: Document ID 1
Run ID: 1
Document: guide_onboarding.pdf
SHA-256: cf22ac557686


In [33]:
# ============================================================
# CELL 23 — INSPECT DOCLING DOCUMENT STRUCTURE
# ============================================================

# Count pages extracted by Docling.
page_count = len(reloaded_doc.pages)

# Count text elements.
text_count = len(reloaded_doc.texts)

# Count tables.
table_count = len(reloaded_doc.tables)

# Count pictures.
image_count = len(reloaded_doc.pictures)

# Display the extraction statistics.
print("Document:", test_document.name)
print("Pages:", page_count)
print("Text elements:", text_count)
print("Tables:", table_count)
print("Pictures:", image_count)

Document: guide_onboarding.pdf
Pages: 2
Text elements: 37
Tables: 2
Pictures: 0


In [34]:
# ============================================================
# CELL 23A — DISPLAY ORIGINAL PDF
# ============================================================

from IPython.display import display, IFrame

# Display the original PDF inside Jupyter.
display(
    IFrame(
        src=test_document.as_uri(),
        width=900,
        height=700
    )
)

In [35]:
# ============================================================
# CELL 24 — UPDATE DOCUMENT EXTRACTION STATISTICS
# ============================================================

# Connect to PostgreSQL.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    # Update the existing document record.
    result = conn.execute(
        """
        UPDATE public.document
        SET
            page_count = %s,
            text_count = %s,
            table_count = %s,
            image_count = %s
        WHERE document_id = %s
          AND run_id = %s
        RETURNING document_id, file_name, status_;
        """,
        (
            page_count,
            text_count,
            table_count,
            image_count,
            document_id,
            run_id
        )
    )

    # Retrieve the updated record.
    updated = result.fetchone()

    # Stop if no matching record was found.
    if updated is None:
        raise ValueError("Document record not found.")

    # Commit the database changes.
    conn.commit()

# Display confirmation.
print("Document updated successfully")
print("Document ID:", updated[0])
print("File:", updated[1])
print("Status:", updated[2])
print("Pages:", page_count)
print("Text elements:", text_count)
print("Tables:", table_count)
print("Pictures:", image_count)

Document updated successfully
Document ID: 1
File: guide_onboarding.pdf
Status: success
Pages: 2
Text elements: 37
Tables: 2
Pictures: 0


In [36]:
# ============================================================
# CELL 25 — SELECT NEXT DOCUMENT FOR PROCESSING
# ============================================================

# Select one document from our existing list.
next_document = next(
    doc for doc in documents
    if doc.name == "grille_salariale_2026.pdf"
)

# Retrieve its previously calculated SHA-256 hash.
next_hash = document_hashes[next_document]

# Verify whether this document was already processed.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    existing = conn.execute(
        """
        SELECT document_id, status_
        FROM public.document
        WHERE run_id = %s
          AND file_hash = %s
          AND status_ = 'success';
        """,
        (run_id, next_hash)
    ).fetchone()

# Display the result.
print("Document:", next_document.name)
print("SHA-256:", next_hash[:12])
print("Previously processed:", existing is not None)

if existing:
    print("SKIP — Already processed successfully")
else:
    print("READY — Document can be processed")

Document: grille_salariale_2026.pdf
SHA-256: 356850c4f1c2
Previously processed: False
READY — Document can be processed


In [38]:
from docling.datamodel.base_models import ConversionStatus

# ============================================================
# CELL 26 — REUSABLE DOCLING CONVERSION FUNCTION
# ============================================================

def convert_document(source_path):

    # 1. Build output paths, preserving the source extension.
    json_output = EXPORT_FOLDER / f"{source_path.name}.json"
    md_output = EXPORT_FOLDER / f"{source_path.name}.md"

    # 2. Prevent accidental overwriting.
    if json_output.exists() or md_output.exists():
        raise FileExistsError(
            f"Existing export detected: {source_path.name}"
        )

    # 3. Measure conversion time.
    start = perf_counter()

    # 4. Convert the source document.
    result = converter.convert(source_path)

    # 5. Verify Docling's conversion status.
    if result.status != ConversionStatus.SUCCESS:
        raise RuntimeError(
            f"Docling conversion failed: {result.status}"
        )

    # 6. Retrieve the structured document and its text.
    converted_doc = result.document
    markdown = converted_doc.export_to_markdown()

    # 7. Reject empty extractions.
    if not markdown.strip():
        raise ValueError(
            f"No usable text extracted: {source_path.name}"
        )

    # 8. Save both formats.
    converted_doc.save_as_json(json_output)
    md_output.write_text(markdown, encoding="utf-8")

    # 9. Reload the JSON and verify content integrity.
    verified_doc = DoclingDocument.load_from_json(json_output)

    if verified_doc.export_to_markdown() != markdown:
        raise ValueError("JSON verification failed")

    # 10. Return the processing results.
    return {
        "source": source_path,
        "json_path": json_output,
        "md_path": md_output,
        "processing_time": perf_counter() - start,
        "page_count": len(verified_doc.pages),
        "text_count": len(verified_doc.texts),
        "table_count": len(verified_doc.tables),
        "image_count": len(verified_doc.pictures),
        "status": "success"
    }

In [39]:
# ============================================================
# CELL 27 — TEST REUSABLE CONVERSION FUNCTION
# ============================================================

# Call the function created in Cell 26.
conversion_data = convert_document(next_document)

# Display the returned processing information.
print("Source:", conversion_data["source"].name)
print("Status:", conversion_data["status"])
print("Processing time:", round(conversion_data["processing_time"], 2), "seconds")
print("Pages:", conversion_data["page_count"])
print("Text elements:", conversion_data["text_count"])
print("Tables:", conversion_data["table_count"])
print("Pictures:", conversion_data["image_count"])

# Verify the exported files.
print("\nEXPORTS")
print("JSON:", conversion_data["json_path"].name)
print("Markdown:", conversion_data["md_path"].name)

2026-10-12 21:50:56,230 MatchingPostProcessor WARNING  3 of 128 pdf cells matched neither a row nor a column band of the 8x6 grid and were dropped from the table
Source: grille_salariale_2026.pdf
Status: success
Processing time: 1.3 seconds
Pages: 1
Text elements: 10
Tables: 2
Pictures: 0

EXPORTS
JSON: grille_salariale_2026.pdf.json
Markdown: grille_salariale_2026.pdf.md


In [40]:
# ============================================================
# CELL 28 — REGISTER CONVERTED DOCUMENT IN POSTGRESQL
# ============================================================

# Retrieve the SHA-256 calculated in Cell 16.
file_hash = document_hashes[next_document]

# Connect to PostgreSQL.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    # Check whether the document is already registered.
    existing = conn.execute(
        """
        SELECT document_id
        FROM public.document
        WHERE run_id = %s
          AND file_hash = %s
          AND status_ = 'success';
        """,
        (run_id, file_hash)
    ).fetchone()

    if existing:
        # Reuse the existing record.
        document_id = existing[0]
        print(f"SKIPPED — Already registered: ID {document_id}")

    else:
        # Insert the complete document-processing record.
        result = conn.execute(
            """
            INSERT INTO public.document (
                run_id,
                file_path,
                export_folder,
                file_name,
                file_format_,
                file_hash,
                status_,
                page_count,
                text_count,
                table_count,
                image_count,
                process_time
            )
            VALUES (
                %s, %s, %s, %s, %s, %s,
                %s, %s, %s, %s, %s, %s
            )
            RETURNING document_id;
            """,
            (
                run_id,
                str(next_document),
                str(EXPORT_FOLDER),
                next_document.name,
                next_document.suffix.lower(),
                file_hash,
                conversion_data["status"],
                conversion_data["page_count"],
                conversion_data["text_count"],
                conversion_data["table_count"],
                conversion_data["image_count"],
                conversion_data["processing_time"]
            )
        )

        document_id = result.fetchone()[0]
        print(f"REGISTERED — Document ID {document_id}")

    # Save changes to PostgreSQL.
    conn.commit()

print("Run ID:", run_id)
print("Document:", next_document.name)

REGISTERED — Document ID 2
Run ID: 1
Document: grille_salariale_2026.pdf


In [41]:
# ============================================================
# CELL 29 — REFRESH DOCUMENT PROCESSING STATUS
# ============================================================

# Retrieve successfully processed document hashes.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    rows = conn.execute(
        """
        SELECT file_hash
        FROM public.document
        WHERE run_id = %s
          AND status_ = 'success';
        """,
        (run_id,)
    ).fetchall()

# Create a set of successfully processed hashes.
successful_hashes = {row[0] for row in rows}

# Refresh the processing lists.
to_process = []
to_skip = []

for document in documents:

    if document_hashes[document] in successful_hashes:
        to_skip.append(document)
    else:
        to_process.append(document)

# Display the updated workload.
print("Run ID:", run_id)
print("Total documents:", len(documents))
print("Successfully registered:", len(to_skip))
print("Remaining documents:", len(to_process))

print("\nREMAINING DOCUMENTS:")
for document in to_process:
    print(" -", document.name)

Run ID: 1
Total documents: 13
Successfully registered: 2
Remaining documents: 11

REMAINING DOCUMENTS:
 - intranet_actualites.html
 - intranet_faq_rh.html
 - newsletter_le_fil_2026-02.pdf
 - organigramme_et_processus_production.pdf
 - plan_strategique_2026-2028.pptx
 - politique_teletravail_2023.docx
 - politique_teletravail_2025.docx
 - politique_teletravail_2025.pdf
 - presentation_veltis.pdf
 - pv_conseil_administration_2026-03-12_scan.pdf
 - rapport_annuel_2025.pdf


In [42]:
# ============================================================
# CELL 30 — AUTOMATED DOCUMENT PROCESSING
# ============================================================

def process_document(source_path, run_id):

    # 1. Retrieve the document's SHA-256 hash.
    file_hash = document_hashes[source_path]

    # 2. Build export paths.
    json_path = EXPORT_FOLDER / f"{source_path.name}.json"
    md_path = EXPORT_FOLDER / f"{source_path.name}.md"

    # 3. Check PostgreSQL for previous successful processing.
    with pg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=db_user,
        password=db_password
    ) as conn:

        existing = conn.execute(
            """
            SELECT document_id
            FROM public.document
            WHERE run_id = %s
              AND file_hash = %s
              AND status_ = 'success';
            """,
            (run_id, file_hash)
        ).fetchone()

    # 4. Reuse an existing record only if both exports exist.
    if existing:
        if json_path.is_file() and md_path.is_file():
            print(f"SKIPPED: {source_path.name}")
            return "SKIPPED"

        # Do not overwrite or silently recreate inconsistent records.
        raise RuntimeError(
            f"Database reports success, but exports are missing: "
            f"{source_path.name}"
        )

    # 5. Prevent overwriting exports from an earlier execution.
    if json_path.exists() or md_path.exists():
        raise FileExistsError(
            f"Exports exist without a matching successful record: "
            f"{source_path.name}"
        )

    # 6. Convert and verify the document.
    data = convert_document(source_path)

    # 7. Register the successful conversion in PostgreSQL.
    with pg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=db_user,
        password=db_password
    ) as conn:

        result = conn.execute(
            """
            INSERT INTO public.document (
                run_id,
                file_path,
                export_folder,
                file_name,
                file_format_,
                file_hash,
                status_,
                page_count,
                text_count,
                table_count,
                image_count,
                process_time
            )
            VALUES (
                %s, %s, %s, %s, %s, %s,
                %s, %s, %s, %s, %s, %s
            )
            RETURNING document_id;
            """,
            (
                run_id,
                str(source_path),
                str(EXPORT_FOLDER),
                source_path.name,
                source_path.suffix.lower(),
                file_hash,
                data["status"],
                data["page_count"],
                data["text_count"],
                data["table_count"],
                data["image_count"],
                data["processing_time"]
            )
        )

        document_id = result.fetchone()[0]
        conn.commit()

    print(f"SUCCESS: {source_path.name} | Document ID: {document_id}")

    return "SUCCESS"

In [43]:
# ============================================================
# CELL 31 — TEST AUTOMATED DOCUMENT PROCESSING
# ============================================================

# Select the third test document.
third_document = next(
    doc for doc in documents
    if doc.name == "presentation_veltis.pdf"
)

# Execute the complete processing pipeline.
try:
    result = process_document(third_document, run_id)

    print("\nProcessing result:", result)

except Exception as error:
    print("PROCESSING FAILED")
    print("Error type:", type(error).__name__)
    print("Details:", error)

SUCCESS: presentation_veltis.pdf | Document ID: 3

Processing result: SUCCESS


In [44]:
# ============================================================
# CELL 32 — AUDIT DOCUMENT PROCESSING STATE
# ============================================================

# Retrieve successful documents from PostgreSQL.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    rows = conn.execute(
        """
        SELECT file_path, file_hash
        FROM public.document
        WHERE run_id = %s
          AND status_ = 'success';
        """,
        (run_id,)
    ).fetchall()

# Use both the source path and SHA-256 to identify records.
registered = {(path, file_hash) for path, file_hash in rows}

print(f"{'DOCUMENT':45} {'DATABASE':12} {'JSON':8} {'MD':8}")
print("-" * 78)

for document in documents:

    # Check the PostgreSQL registration.
    db_ok = (
        str(document),
        document_hashes[document]
    ) in registered

    # Check the two exported files.
    json_file = EXPORT_FOLDER / f"{document.name}.json"
    md_file = EXPORT_FOLDER / f"{document.name}.md"

    json_ok = json_file.is_file()
    md_ok = md_file.is_file()

    print(
        f"{document.name:45} "
        f"{'SUCCESS' if db_ok else 'MISSING':12} "
        f"{'YES' if json_ok else 'NO':8} "
        f"{'YES' if md_ok else 'NO':8}"
    )

DOCUMENT                                      DATABASE     JSON     MD      
------------------------------------------------------------------------------
grille_salariale_2026.pdf                     SUCCESS      YES      YES     
guide_onboarding.pdf                          SUCCESS      YES      YES     
intranet_actualites.html                      MISSING      NO       NO      
intranet_faq_rh.html                          MISSING      NO       NO      
newsletter_le_fil_2026-02.pdf                 MISSING      NO       NO      
organigramme_et_processus_production.pdf      MISSING      NO       NO      
plan_strategique_2026-2028.pptx               MISSING      NO       NO      
politique_teletravail_2023.docx               MISSING      NO       NO      
politique_teletravail_2025.docx               MISSING      NO       NO      
politique_teletravail_2025.pdf                MISSING      NO       NO      
presentation_veltis.pdf                       SUCCESS      YES      YES   

In [46]:
# ============================================================
# CELL 33 — VALIDATE EXISTING DOCLING EXPORTS
# ============================================================

def validate_exports(source_path):

    # Build paths using the complete original filename.
    json_file = EXPORT_FOLDER / f"{source_path.name}.json"
    md_file = EXPORT_FOLDER / f"{source_path.name}.md"

    # Both exported files must exist.
    if not json_file.is_file() or not md_file.is_file():
        return False

    try:
        # Reconstruct the document from JSON.
        loaded_doc = DoclingDocument.load_from_json(json_file)

        # Extract Markdown from the reconstructed document.
        extracted_text = loaded_doc.export_to_markdown()

        # Read the Markdown file saved on disk.
        saved_text = md_file.read_text(encoding="utf-8")

        # Verify that text exists and both versions agree.
        return bool(extracted_text.strip()) and extracted_text == saved_text

    except Exception as error:
        print(f"Validation error for {source_path.name}: {error}")
        return False

    # ============================================================
# TEST — VERIFY OUR THREE PROCESSED DOCUMENTS
# ============================================================

for name in [
    "guide_onboarding.pdf",
    "grille_salariale_2026.pdf",
    "presentation_veltis.pdf"
]:
    source = next(doc for doc in documents if doc.name == name)

    print(
        source.name,
        "| Valid exports:",
        validate_exports(source)
    )

guide_onboarding.pdf | Valid exports: True
grille_salariale_2026.pdf | Valid exports: True
presentation_veltis.pdf | Valid exports: True


In [47]:
# ============================================================
# CELL 34 — RESUMABLE DOCUMENT PROCESSING
# ============================================================

def process_document(source_path, run_id):

    # Identify the source document.
    file_hash = document_hashes[source_path]

    # Determine the output filenames.
    json_path = EXPORT_FOLDER / f"{source_path.name}.json"
    md_path = EXPORT_FOLDER / f"{source_path.name}.md"

    # Check whether PostgreSQL already records a successful conversion.
    with pg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=db_user,
        password=db_password
    ) as conn:

        existing = conn.execute(
            """
            SELECT document_id
            FROM public.document
            WHERE run_id = %s
              AND file_path = %s
              AND file_hash = %s
              AND status_ = 'success';
            """,
            (run_id, str(source_path), file_hash)
        ).fetchone()

    # CASE 1 — Already registered.
    if existing:

        if not validate_exports(source_path):
            raise RuntimeError(
                f"Registered document has invalid exports: {source_path.name}"
            )

        print(f"SKIPPED: {source_path.name}")
        return "SKIPPED"

    # CASE 2 — No database record, but exports exist.
    if json_path.exists() or md_path.exists():

        # Validate the exports before considering recovery.
        if not validate_exports(source_path):
            raise RuntimeError(
                f"Incomplete or invalid exports: {source_path.name}"
            )

        # Without proof of the source version, do not register old exports.
        raise RuntimeError(
            f"Valid exports exist without a matching database record: "
            f"{source_path.name}. Provenance requires verification."
        )

    # CASE 3 — New document requiring conversion.
    data = convert_document(source_path)

    # Register successful processing.
    with pg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=db_user,
        password=db_password
    ) as conn:

        result = conn.execute(
            """
            INSERT INTO public.document (
                run_id,
                file_path,
                export_folder,
                file_name,
                file_format_,
                file_hash,
                status_,
                page_count,
                text_count,
                table_count,
                image_count,
                process_time
            )
            VALUES (
                %s, %s, %s, %s, %s, %s,
                %s, %s, %s, %s, %s, %s
            )
            RETURNING document_id;
            """,
            (
                run_id,
                str(source_path),
                str(EXPORT_FOLDER),
                source_path.name,
                source_path.suffix.lower(),
                file_hash,
                data["status"],
                data["page_count"],
                data["text_count"],
                data["table_count"],
                data["image_count"],
                data["processing_time"]
            )
        )

        document_id = result.fetchone()[0]
        conn.commit()

    print(f"SUCCESS: {source_path.name} | Document ID: {document_id}")

    return "SUCCESS"

In [48]:
# ============================================================
# CELL 35 — TEST IDEMPOTENCY
# ============================================================

result = process_document(test_document, run_id)

print("Result:", result)

SKIPPED: guide_onboarding.pdf
Result: SKIPPED


In [49]:
# ============================================================
# CELL 36 — CHECK REMAINING DOCUMENTS
# ============================================================

from collections import Counter

# Count how many times each filename appears in the corpus.
name_counts = Counter(doc.name for doc in documents)

# Detect filenames that appear more than once.
duplicate_names = [
    name for name, count in name_counts.items()
    if count > 1
]

# Refresh the list of documents requiring processing.
remaining = [
    doc for doc in documents
    if document_hashes[doc] not in successful_hashes
]

# Display the inventory.
print("Total documents:", len(documents))
print("Duplicate filenames:", duplicate_names)

print("\nDOCUMENTS:")
for doc in documents:
    print(f"{doc.suffix.lower():6} | {doc.relative_to(CORPUS)}")

# Prevent batch processing if filename collisions exist.
if duplicate_names:
    raise RuntimeError(
        "Duplicate filenames detected. "
        "Export naming must be corrected before continuing."
    )

Total documents: 13
Duplicate filenames: []

DOCUMENTS:
.pdf   | grille_salariale_2026.pdf
.pdf   | guide_onboarding.pdf
.html  | intranet_actualites.html
.html  | intranet_faq_rh.html
.pdf   | newsletter_le_fil_2026-02.pdf
.pdf   | organigramme_et_processus_production.pdf
.pptx  | plan_strategique_2026-2028.pptx
.docx  | politique_teletravail_2023.docx
.docx  | politique_teletravail_2025.docx
.pdf   | politique_teletravail_2025.pdf
.pdf   | presentation_veltis.pdf
.pdf   | pv_conseil_administration_2026-03-12_scan.pdf
.pdf   | rapport_annuel_2025.pdf


In [50]:
# ============================================================
# CELL 37 — TEST DOCX PROCESSING
# ============================================================

# Select the DOCX document from our existing inventory.
docx_document = next(
    doc for doc in documents
    if doc.name == "politique_teletravail_2025.docx"
)

# Run the complete pipeline.
try:
    result = process_document(docx_document, run_id)

    print("\nProcessing result:", result)

except Exception as error:
    print("\nPROCESSING FAILED")
    print("Error type:", type(error).__name__)
    print("Details:", error)

SUCCESS: politique_teletravail_2025.docx | Document ID: 4

Processing result: SUCCESS


In [51]:
# ============================================================
# CELL 38 — TEST HTML PROCESSING
# ============================================================

# Select the HTML document from our existing inventory.
html_document = next(
    doc for doc in documents
    if doc.name == "intranet_faq_rh.html"
)

# Execute our reusable processing function.
try:
    result = process_document(html_document, run_id)

    print("\nProcessing result:", result)

except Exception as error:
    print("\nPROCESSING FAILED")
    print("Error type:", type(error).__name__)
    print("Details:", error)

SUCCESS: intranet_faq_rh.html | Document ID: 5

Processing result: SUCCESS


In [52]:
# ============================================================
# CELL 39 — TEST PPTX PROCESSING
# ============================================================

# Select the PowerPoint document from our corpus.
pptx_document = next(
    doc for doc in documents
    if doc.name == "plan_strategique_2026-2028.pptx"
)

# Process it using our existing function.
try:
    result = process_document(pptx_document, run_id)

    print("\nProcessing result:", result)

except Exception as error:
    print("\nPROCESSING FAILED")
    print("Error type:", type(error).__name__)
    print("Details:", error)

SUCCESS: plan_strategique_2026-2028.pptx | Document ID: 6

Processing result: SUCCESS


In [53]:
# ============================================================
# CELL 40 — PROCESS REMAINING DOCUMENTS
# ============================================================

# Store the results of this batch.
batch_results = {}

# Process all selected documents.
for document in documents:

    try:
        # Reuse our existing function from Cell 34.
        result = process_document(document, run_id)

        # Record the outcome.
        batch_results[document.name] = result

    except Exception as error:

        # Record the failure without stopping the batch.
        batch_results[document.name] = "FAILED"

        print(f"FAILED: {document.name}")
        print(f"Error: {type(error).__name__}: {error}")

# Display the batch summary.
print("\n===== BATCH SUMMARY =====")

for name, status in batch_results.items():
    print(f"{status:8} | {name}")

print("\nTotal examined:", len(batch_results))
print("Successful now:", list(batch_results.values()).count("SUCCESS"))
print("Already processed:", list(batch_results.values()).count("SKIPPED"))
print("Failed:", list(batch_results.values()).count("FAILED"))

SKIPPED: grille_salariale_2026.pdf
SKIPPED: guide_onboarding.pdf
SUCCESS: intranet_actualites.html | Document ID: 7
SKIPPED: intranet_faq_rh.html
SUCCESS: newsletter_le_fil_2026-02.pdf | Document ID: 8
SUCCESS: organigramme_et_processus_production.pdf | Document ID: 9
SKIPPED: plan_strategique_2026-2028.pptx
SUCCESS: politique_teletravail_2023.docx | Document ID: 10
SKIPPED: politique_teletravail_2025.docx
SUCCESS: politique_teletravail_2025.pdf | Document ID: 11
SKIPPED: presentation_veltis.pdf
SUCCESS: pv_conseil_administration_2026-03-12_scan.pdf | Document ID: 12
SUCCESS: rapport_annuel_2025.pdf | Document ID: 13

===== BATCH SUMMARY =====
SKIPPED  | grille_salariale_2026.pdf
SKIPPED  | guide_onboarding.pdf
SUCCESS  | intranet_actualites.html
SKIPPED  | intranet_faq_rh.html
SUCCESS  | newsletter_le_fil_2026-02.pdf
SUCCESS  | organigramme_et_processus_production.pdf
SKIPPED  | plan_strategique_2026-2028.pptx
SUCCESS  | politique_teletravail_2023.docx
SKIPPED  | politique_teletravail

In [54]:
# ============================================================
# CELL 41 — AUDIT DOCUMENT EXTRACTION
# ============================================================

# Retrieve registered documents from PostgreSQL.
with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    rows = conn.execute(
        """
        SELECT
            document_id,
            file_name,
            status_,
            page_count,
            text_count,
            table_count,
            image_count
        FROM public.document
        WHERE run_id = %s
        ORDER BY document_id;
        """,
        (run_id,)
    ).fetchall()

# Display the registered documents.
print(f"{'ID':<4} {'DOCUMENT':<47} {'STATUS':<10} {'PAGES':<7} {'TEXT':<7}")

for row in rows:
    doc_id, name, status, pages, texts, tables, images = row

    print(
        f"{doc_id:<4} "
        f"{name:<47} "
        f"{status:<10} "
        f"{str(pages):<7} "
        f"{str(texts):<7}"
    )

# Verify the extracted content of the scanned PDF.
scan_name = "pv_conseil_administration_2026-03-12_scan.pdf"

scan_json = EXPORT_FOLDER / f"{scan_name}.json"

scan_doc = DoclingDocument.load_from_json(scan_json)

scan_text = scan_doc.export_to_markdown()

print("\n===== SCANNED PDF INSPECTION =====")
print("Document:", scan_name)
print("Extracted characters:", len(scan_text))
print("Non-empty text:", bool(scan_text.strip()))

print("\nTEXT PREVIEW:")
print(scan_text[:1500])

ID   DOCUMENT                                        STATUS     PAGES   TEXT   
1    guide_onboarding.pdf                            success    2       37     
2    grille_salariale_2026.pdf                       success    1       10     
3    presentation_veltis.pdf                         success    2       36     
4    politique_teletravail_2025.docx                 success    0       19     
5    intranet_faq_rh.html                            success    0       32     
6    plan_strategique_2026-2028.pptx                 success    7       35     
7    intranet_actualites.html                        success    0       27     
8    newsletter_le_fil_2026-02.pdf                   success    1       23     
9    organigramme_et_processus_production.pdf        success    2       48     
10   politique_teletravail_2023.docx                 success    0       17     
11   politique_teletravail_2025.pdf                  success    1       19     
12   pv_conseil_administration_2026-03-1

In [56]:
# ============================================================
# CELL 42 — RECONCILE AND CLOSE RUN 1
# ============================================================

with pg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=db_user,
    password=db_password
) as conn:

    # Retrieve the registered successful documents.
    rows = conn.execute(
        """
        SELECT file_path, file_hash
        FROM public.document
        WHERE run_id = %s
          AND status_ = 'success';
        """,
        (run_id,)
    ).fetchall()

    # Identify documents using both their paths and hashes.
    registered = set(rows)

    # Construct the expected set from our source documents.
    expected = {
        (str(doc), document_hashes[doc])
        for doc in documents
    }

    # Compare actual records against the expected documents.
    missing = expected - registered
    unexpected = registered - expected

    print("Expected documents:", len(expected))
    print("Registered documents:", len(registered))
    print("Missing documents:", len(missing))
    print("Unexpected documents:", len(unexpected))

    # Stop if the records do not match exactly.
    if missing or unexpected or len(rows) != len(expected):
        raise RuntimeError(
            "Run reconciliation failed. "
            "The run will remain open."
        )

    # Verify that the existing run has not already been closed.
    current_run = conn.execute(
        """
        SELECT end_time
        FROM public.run
        WHERE run_id = %s;
        """,
        (run_id,)
    ).fetchone()

    if current_run is None:
        raise RuntimeError("Run not found.")

    if current_run[0] is not None:
        print("Run already closed. No update necessary.")

    else:
        # Update processing totals and completion time.
        conn.execute(
            """
            UPDATE public.run
            SET
                doc_processed = %s,
                doc_ignored = %s,
                doc_failed = %s,
                end_time = NOW()
            WHERE run_id = %s;
            """,
            (len(expected), 0, 0, run_id)
        )

        conn.commit()

        print("Run closed successfully:", run_id)

Expected documents: 13
Registered documents: 13
Missing documents: 0
Unexpected documents: 0
Run already closed. No update necessary.
